In [ ]:
5+5

# How far is the true airport from the last point of the flight

# Is current flight algoirhtm finding the truely last message in sequence

In [1]:
from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.flights.flight_type import get_flights
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
from datetime import date
import polars as pl
target_date = date(2026, 6, 1)
df_sfdps = get_sfdps_flights_day(target_date)
df_test = get_flights(target_date)
df_test = df_test.filter(pl.col("pia") | pl.col("ladd"))
df_test = df_test.filter(pl.col("icao").is_in(df_sfdps.select("icao").to_series().implode()))
df_adsb = read_adsb(target_date)
df_adsb = df_adsb.filter(pl.col("icao").is_in(df_test.select("icao").to_series().implode()))

In [5]:
len(df_sfdps.get_column("icao").unique().to_list())

8553

In [3]:
from data_engineering.flights.flight_type import add_lat_lon_to_airport_ident_columns


add_lat_lon_to_airport_ident_columns(df_sfdps).filter(pl.col("landing_airport_lat").is_null() | pl.col("landing_airport_lon").is_null()).select(["icao", "landing_airport_ident"]).unique().sort("icao")

icao,landing_airport_ident
str,str
"""3a2133""","""EE1"""
"""a00302""","""HVC"""
"""a00318""","""FSM"""
"""a005f2""","""RNO"""
"""a00603""","""AFJ"""
…,…
"""ade0cc""","""NFD"""
"""ade840""","""DTO"""
"""adef77""","""9G3"""


In [19]:
import polars as pl

max_future_gap_seconds = 5 * 60
adsb_cols = [
    col
    for col in ["icao", "time", "lat", "lon", "ground_speed_kt", "on_ground", "baro_altitude_ft"]
    if col in df_adsb.columns
]

future_adsb = (
    df_adsb
    .select(adsb_cols)
    .rename(
        {
            "time": "future_adsb_time",
            "lat": "future_adsb_lat",
            "lon": "future_adsb_lon",
            "ground_speed_kt": "future_adsb_ground_speed_kt",
            "on_ground": "future_adsb_on_ground",
            "baro_altitude_ft": "future_adsb_baro_altitude_ft",
        }
    )
    .filter(pl.col("future_adsb_time").is_not_null())
    .sort(["icao", "future_adsb_time"])
)

future_adsb_value_cols = [col for col in future_adsb.columns if col != "icao"]
endpoint_time_summary = df_test.select(
    pl.len().alias("flight_count"),
    pl.col("first_message_time").is_not_null().sum().alias("has_first_message_time"),
    pl.col("last_message_time").is_not_null().sum().alias("has_last_message_time"),
    (
        (pl.col("takeoff_time") - pl.col("first_message_time"))
        .dt.total_milliseconds()
        / 1000
    ).mean().alias("avg_first_message_to_takeoff_seconds"),
    (
        (pl.col("last_message_time") - pl.col("landing_time"))
        .dt.total_milliseconds()
        / 1000
    ).mean().alias("avg_landing_to_last_message_seconds"),
)

landing_future_adsb = (
    df_test
    .with_row_index("flight_idx")
    .with_columns(
        (
            (pl.col("takeoff_time") - pl.col("first_message_time"))
            .dt.total_milliseconds()
            / 1000
        ).alias("first_message_to_takeoff_seconds"),
        (
            (pl.col("last_message_time") - pl.col("landing_time"))
            .dt.total_milliseconds()
            / 1000
        ).alias("landing_to_last_message_seconds"),
        (
            pl.col("last_message_time") + pl.duration(milliseconds=1)
        ).alias("last_message_future_search_time"),
    )
    .filter(pl.col("last_message_future_search_time").is_not_null())
    .sort(["icao", "last_message_future_search_time"])
    .join_asof(
        future_adsb,
        left_on="last_message_future_search_time",
        right_on="future_adsb_time",
        by="icao",
        strategy="forward",
        check_sortedness=False,
    )
    .with_columns(
        (
            (pl.col("future_adsb_time") - pl.col("last_message_time"))
            .dt.total_milliseconds()
            / 1000
        ).alias("last_message_to_future_adsb_seconds")
    )
    .with_columns(
        pl.when(pl.col("last_message_to_future_adsb_seconds") <= max_future_gap_seconds)
        .then(pl.col("last_message_to_future_adsb_seconds"))
        .otherwise(None)
        .alias("last_message_to_future_adsb_seconds")
    )
    .with_columns(
        [
            pl.when(pl.col("last_message_to_future_adsb_seconds").is_not_null())
            .then(pl.col(col))
            .otherwise(None)
            .alias(col)
            for col in future_adsb_value_cols
        ]
    )
    .with_columns(
        (
            pl.col("last_message_to_future_adsb_seconds") / 60
        ).alias("last_message_to_future_adsb_minutes")
    )
    .select(
        [
            "flight_idx",
            "icao",
            "takeoff_time",
            "first_message_time",
            "first_message_to_takeoff_seconds",
            "landing_time",
            "last_message_time",
            "landing_to_last_message_seconds",
            "takeoff_airport_ident",
            "landing_airport_ident",
            "last_message_to_future_adsb_seconds",
            "last_message_to_future_adsb_minutes",
            *future_adsb_value_cols,
        ]
    )
)

landing_future_adsb_summary = landing_future_adsb.select(
    pl.len().alias("flights_with_last_message_time"),
    pl.col("last_message_to_future_adsb_seconds").is_not_null().sum().alias("has_future_adsb_after_last_message_within_5m"),
    pl.col("last_message_to_future_adsb_seconds").mean().alias("avg_seconds"),
    pl.col("last_message_to_future_adsb_seconds").quantile(0.50).alias("p50_seconds"),
    pl.col("last_message_to_future_adsb_seconds").quantile(0.95).alias("p95_seconds"),
    pl.col("last_message_to_future_adsb_seconds").max().alias("max_seconds"),
)

display(endpoint_time_summary)
display(landing_future_adsb_summary)
landing_future_adsb.sort("last_message_to_future_adsb_seconds", descending=True, nulls_last=True)


flight_count,has_first_message_time,has_last_message_time,avg_first_message_to_takeoff_seconds,avg_landing_to_last_message_seconds
u32,u32,u32,f64,f64
1391,1391,1391,258.310334,35.629382


flights_with_last_message_time,has_future_adsb_after_last_message_within_5m,avg_seconds,p50_seconds,p95_seconds,max_seconds
u32,u32,f64,f64,f64,f64
1391,288,11.162153,3.52,49.84,254.46


flight_idx,icao,takeoff_time,first_message_time,first_message_to_takeoff_seconds,landing_time,last_message_time,landing_to_last_message_seconds,takeoff_airport_ident,landing_airport_ident,last_message_to_future_adsb_seconds,last_message_to_future_adsb_minutes,future_adsb_time,future_adsb_lat,future_adsb_lon,future_adsb_ground_speed_kt,future_adsb_on_ground,future_adsb_baro_altitude_ft
u32,str,datetime[ms],datetime[ms],f64,datetime[ms],datetime[ms],f64,str,str,f64,f64,datetime[ms],f64,f64,f32,bool,i32
675,"""a519da""",2026-06-01 01:15:39.030,2026-06-01 01:15:38.090,0.94,2026-06-01 01:38:37.050,2026-06-01 01:40:46.320,129.27,"""KPOC""","""KLGB""",254.46,4.241,2026-06-01 01:45:00.780,33.813904,-118.159977,null,false,300
1071,"""a95db5""",2026-06-01 21:00:53.100,2026-06-01 21:00:45.160,7.94,2026-06-01 23:27:51.860,2026-06-01 23:29:54.660,122.8,"""KSGR""","""KSLC""",223.98,3.733,2026-06-01 23:33:38.640,40.777737,-111.960373,8.8,true,null
420,"""a29982""",2026-06-01 13:43:21.550,2026-06-01 13:43:07.460,14.09,2026-06-01 13:58:19.020,2026-06-01 14:01:13.790,174.77,"""KDYL""","""KTTN""",218.77,3.646167,2026-06-01 14:04:52.560,40.282516,-74.814438,null,true,null
301,"""a23a09""",2026-06-01 15:46:09.780,2026-06-01 15:45:25.910,43.87,2026-06-01 16:33:48.140,2026-06-01 16:36:59.720,191.58,"""KCRQ""","""KLGB""",141.77,2.362833,2026-06-01 16:39:21.490,33.813091,-118.151443,10.2,true,null
818,"""a66103""",2026-06-01 09:45:11.960,2026-06-01 09:30:43.040,868.92,2026-06-01 10:27:59.080,2026-06-01 10:31:51.240,232.16,"""KBCB""","""KORF""",134.88,2.248,2026-06-01 10:34:06.120,36.886677,-76.200427,0.0,true,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1386,"""adedaa""",2026-06-01 22:44:09.810,2026-06-01 22:50:48.320,-398.51,2026-06-01 23:45:35.530,2026-06-01 23:45:35.530,0.0,"""KCDN""","""KRBW""",null,null,null,null,null,null,null,null
1387,"""adf567""",2026-06-01 19:03:17.430,2026-06-01 19:02:21.860,55.57,2026-06-01 19:58:36,2026-06-01 19:58:36,0.0,"""KAPC""","""KVNY""",null,null,null,null,null,null,null,null
1388,"""adf567""",2026-06-01 23:34:42.260,2026-06-01 23:23:23.950,678.31,2026-06-02 00:17:48.370,2026-06-02 00:20:01.410,133.04,"""KVNY""","""KLAS""",null,null,null,null,null,null,null,null


In [20]:
df_sfdps.filter(pl.col("icao") == "a519da")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a519da""","""CXK443""","""N428FF""",2026-06-01 01:15:00,"""KPOC""",2026-06-01 01:38:00,"""LGB""",false,true,false,false,"""C72R""","""DHS OPERATIONS LLC""","""CESSNA 172R Cutlass RG""",""""""


In [21]:
df_test.filter(pl.col("icao") == "a519da")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str
"""a519da""","""""","""""",2026-06-01 01:15:39.030,"""KPOC""",1.545991,2026-06-01 01:38:37.050,"""KLGB""",0.386593,2026-06-01 01:15:38.090,34.091693,-117.798786,1600,1450,2026-06-01 01:40:46.320,33.813091,-118.15056,null,null,439,null,false,true,false,false,"""C72R""","""DHS OPERATIONS LLC""","""CESSNA 172R Cutlass RG""","""A1"""
"""a519da""","""""","""""",2026-06-01 02:58:41.350,"""KLGB""",0.833091,2026-06-01 03:19:08.440,"""KPOC""",0.409314,2026-06-01 02:58:39.220,33.813961,-118.158365,null,null,2026-06-01 03:19:08.440,34.091537,-117.786442,1100,925,null,-86,false,true,false,false,"""C72R""","""DHS OPERATIONS LLC""","""CESSNA 172R Cutlass RG""","""A1"""
"""a519da""","""""","""""",2026-06-01 20:29:26.750,"""KPOC""",0.929505,2026-06-01 21:13:44.460,"""KPOC""",0.22742,2026-06-01 20:29:15.660,34.091693,-117.792091,1300,null,2026-06-01 21:13:44.460,34.091217,-117.784424,1000,875,289,-136,false,true,false,false,"""C72R""","""DHS OPERATIONS LLC""","""CESSNA 172R Cutlass RG""","""A1"""
"""a519da""","""""","""""",2026-06-01 23:14:25.010,"""KPOC""",0.993326,2026-06-01 23:39:46.620,"""KPOC""",0.233514,2026-06-01 23:14:14.210,34.090622,-117.79272,1400,1300,2026-06-01 23:40:04.940,34.090988,-117.784424,1000,875,289,-136,false,true,false,false,"""C72R""","""DHS OPERATIONS LLC""","""CESSNA 172R Cutlass RG""","""A1"""


In [10]:
df_adsb.filter(pl.col("icao") == "a9a266").get_column("category").value_counts()

category,count
str,u32
null,1067
"""A1""",353


In [ ]:
df_flights_0 = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/v4/adsblol/year=2026/month=03/day=01/flights.parquet')
df_flights_1 = pl.read_parquet('/Volumes/T2-SSD/planequery/data/flights/trad-model/v5/adsblol/year=2026/month=03/day=01/flights.parquet')

endpoint_time_cols = [
    "icao",
    "takeoff_time",
    "first_message_time",
    "landing_time",
    "last_message_time",
    "first_lat",
    "first_lon",
    "last_lat",
    "last_lon",
]

df_flights_1.filter(pl.col("icao") == "a0001a").select(endpoint_time_cols)


In [ ]:
[
    col
    for col in df_flights_1.columns
    if col in {"takeoff_time", "first_message_time", "landing_time", "last_message_time"}
]
